# National Research Platform (NRP)
### Chris Endemann, endemann@wisc.edu
### [Nexus version](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/NRP-Nautilus.html)
### Categories
- Compute
- Notebooks
- Code-along
- GPU
- LLM
- API
- Kubernetes
- Jupyter
- Education
- NSF


**Free frontier-LLM endpoints for researchers** — plus free GPU compute on a multi-site Kubernetes cluster. The [National Research Platform (NRP)](https://nrp.ai/) is NSF-supported and open at no cost to researchers and educators at [InCommon](https://incommon.org/)-affiliated institutions (UW–Madison is in). Usage is governed by the [Acceptable Use Policy](https://nrp.ai/NRP-AUP.pdf) and per-user fair-use limits.

For ML/AI work, NRP gives you:

1. **Hosted LLM endpoints** — call open-weight models through an OpenAI-compatible API at `https://ellm.nrp-nautilus.io/v1`. No GPU on your side; NRP runs the inference. See the [model table](#available-llms-and-fair-use-limits) below.
2. **General compute** for everything else — your own GPU pods (RTX 2080 Ti / 3090 / 4090, A10, A100) for training, fine-tuning, or non-LLM work. Most users go through the hosted [JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/) or [Coder](https://nrp.ai/documentation/userdocs/coder/coder/) (VS Code in browser); classrooms can request private JupyterHub deployments via [support@nationalresearchplatform.org](mailto:support@nationalresearchplatform.org).

## Get started

1. **Sign in** at [nrp.ai](https://nrp.ai/) via CILogon. *Pick one identity provider and stick with it* — authentik binds your account to the first IdP you use, and switching later gives a "Permission denied" error. I typically use the Google login option.
2. **Join *Matrix*** ([nrp.ai/contact](https://nrp.ai/contact/)) for support and account promotion. Admin requests, LLM-flag enablement, and most help happen in the **Nautilus Support** channel — DMs to admins are rejected.
3. **Get into a namespace** — required for *both* LLM API and GPU/compute access (an account alone has no resources). Three patterns:
   - **Solo researcher / faculty / postdoc:** ask in Nautilus Support to be promoted to admin, then create your own namespace at [nrp.ai/namespaces](https://nrp.ai/namespaces). NRP's convention is `<institution>-<group>` (e.g., `wisc-ml-marathon`); admins may rename to fit. Mention LLM use in your request and they'll enable the LLM flag at the same time.
   - **Lab or research group:** the PI becomes the namespace admin and adds members.
   - **Student:** ask your advisor to add you to their existing namespace.
4. **Generate an LLM token** at [nrp.ai/llmtoken](https://nrp.ai/llmtoken). If you forgot to mention LLM use up front and your namespace isn't flagged for it, ask in Nautilus Support — it's a one-line toggle.
5. **For general compute** (anything beyond the LLM endpoint — training, fine-tuning, your own notebooks): most users go through the hosted [JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/) or [Coder](https://nrp.ai/documentation/userdocs/coder/coder/) (browser-based, no install). Only if you want to write Kubernetes YAML directly: install `kubectl` + [`kubelogin`](https://github.com/int128/kubelogin) and drop the [NRP config](https://nrp.ai/config) at `~/.kube/config` — full walkthrough in [Getting Started](https://nrp.ai/documentation/userdocs/start/getting-started/).

Read the [Acceptable Use Policy](https://nrp.ai/NRP-AUP.pdf) and [Cluster Policies](https://nrp.ai/documentation/userdocs/start/policies/) before submitting workloads — there are real resource-utilization rules and you can be banned for violating them.

## Available LLMs and fair-use limits

The catalog rotates as the open-weights frontier moves (`main` = generally supported, `evaluating` = may change). Full per-model cards on the [models page](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/); see [lifecycle & changelog](https://nrp.ai/documentation/userdocs/ai/llm-managed/lifecycle/) before pinning a model name in production scripts.

| Model | Status | Params | Context | Modalities | Max concurrent |
|-------|--------|-------:|--------:|------------|---------------:|
| `qwen3` | main | 397B | 1.01M | image, video | 16 |
| `qwen3-small` | main | 27B | 1.01M | image, video | 8 |
| `gpt-oss` | main | 120B | 131K | text | 16 |
| `gemma` | main | 31B | 262K | image, video | 8 |
| `qwen3-embedding` | main | 8B | — | image, video | 16 |
| `gemma-small` | evaluating | ~8B | 131K | image, video, audio | 8 |
| `kimi` | evaluating | 1T | 262K | image, video | 2 |
| `glm-5` | evaluating | 744B | 203K | text | 4 |
| `minimax-m2` | evaluating | 230B | 205K | text | 8 |
| `olmo` | evaluating | 32B | 65K | text | 8 |

"Max concurrent" = how many requests you can have **open at the same instant** under one access token. Not per minute, not per day — just simultaneously. A single-threaded script that sends one request, waits, then sends the next has concurrency 1 and never hits the limit, however long it runs. Parallel/`asyncio` code that fires N requests at once does. There is **no published requests-per-minute, tokens-per-minute, or daily token cap** — concurrency is the only quantified throttle.

Requests using ≥35% of a model's context drop to 1 concurrent request, and the sum of in-flight context must stay under 35%. For higher-volume sessions, contact admins in advance.

Point any OpenAI-compatible client at NRP's API with your token: desktop apps (Chatbox, Cherry Studio), coding CLIs (Claude Code, OpenCode, Crush, Kimi CLI, Copilot CLI), or Python — see [client configurations](https://nrp.ai/documentation/userdocs/ai/llm-managed/client-configs/) and [API access docs](https://nrp.ai/documentation/userdocs/ai/llm-managed/api-access/). NRP also hosts [browser chat UIs](https://nrp.ai/documentation/userdocs/ai/llm-managed/chat-interfaces/) (Open WebUI, LibreChat), though as of May 2026 they've been intermittently unreliable.

## Calling the LLM endpoint from Python

A short code-along. To run it yourself, you'll need:

1. **An LLM token.** Visit [nrp.ai/llmtoken](https://nrp.ai/llmtoken) (you need to be in an LLM-flagged namespace — see [Get started](#get-started) above) and click to mint one. The page shows a long opaque string — copy it and treat it like a password. Don't paste it into source code or commit it.
2. **A local `.env` file** so the code below can read the token from an environment variable instead of you typing it in. In the same directory as this notebook, create a file called `.env`:

   ```
   # .env  (do not commit this file)
   NRP_LLM_TOKEN=paste-your-long-token-string-here
   ```

   Then add `.env` to your `.gitignore`. The `python-dotenv` package in the install line below reads `.env` automatically into `os.environ`.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-DataScience/ML-X-Nexus/blob/main/Toolbox/Compute/NRP-Nautilus.ipynb)

**Install and configure the client.**

In [ ]:
# pip install --quiet openai python-dotenv requests
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(
    api_key=os.environ["NRP_LLM_TOKEN"],
    base_url="https://ellm.nrp-nautilus.io/v1",
)

**List available models.** Expect `qwen3`, `gpt-oss`, `kimi`, `gemma`, `qwen3-embedding`, etc. — the live set rotates.

In [ ]:
for m in client.models.list().data:
    print(m.id)

**A chat completion.** `gpt-oss` is a good first call — text-only, low GPU footprint, stable across model updates.

In [ ]:
completion = client.chat.completions.create(
    model="gpt-oss",
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant for a graduate ML course."},
        {"role": "user", "content": "In 2-3 sentences, when should I use LoRA instead of full fine-tuning?"},
    ],
)
print(completion.choices[0].message.content)

If your client needs `max_tokens`, set it to ~1/3–1/4 of the model's context — it caps output, not total context.

**Toggle reasoning mode.** Several models default to "thinking" mode, which adds latency. Disable via `extra_body`:

In [ ]:
completion = client.chat.completions.create(
    model="qwen3-small",
    messages=[{"role": "user", "content": "One-line definition of overfitting."}],
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(completion.choices[0].message.content)

Exact keys vary by model — see each [model card](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) for `glm-5`, `kimi`, etc.

**Multimodal image input.** `gemma`, `qwen3`, `qwen3-small`, `gemma-small`, and `kimi` accept images as `image_url` content blocks (URL or base64 data URI). `gemma-small` is the only catalogued model that also accepts **audio** (ASR / speech-to-text).

In [ ]:
import base64, requests

img = requests.get("https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg", timeout=30).content
b64 = base64.b64encode(img).decode("ascii")

completion = client.chat.completions.create(
    model="gemma",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "In one sentence: what animal and what is it doing?"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ],
    }],
)
print(completion.choices[0].message.content)

**Embeddings for RAG.** `qwen3-embedding` is the embedding model; do not call it for chat completions.

In [ ]:
emb = client.embeddings.create(
    model="qwen3-embedding",
    input=[
        "Retrieval-augmented generation pairs a vector store with an LLM.",
        "LoRA inserts low-rank adapter matrices into a frozen base model.",
    ],
)
print(f"dim={len(emb.data[0].embedding)}, first 5={emb.data[0].embedding[:5]}")

Vectors plug into any vector DB (Chroma, Qdrant, pgvector, etc.).

**Cache isolation for sensitive prompts.** NRP caches prompts and responses across users sharing a tenant. If yours could be sensitive, pass a `cache_salt` (≥256 random bits, base64) that only you know:

In [ ]:
project_salt = base64.b64encode(os.urandom(32)).decode("ascii")

completion = client.chat.completions.create(
    model="gpt-oss",
    messages=[{"role": "user", "content": "Summarize this internal memo: ..."}],
    extra_body={"cache_salt": project_salt},
)
print(completion.choices[0].message.content)

## Cluster compute essentials

NRP is Kubernetes — workloads are described in YAML. Key rules:

- **Containers are stateless.** Use persistent volumes for anything you can't lose.
- **Interactive pods**: 6 hr max, capped at 2 GPUs / 32 GB RAM / 16 CPU (see [Cluster Policies](https://nrp.ai/documentation/userdocs/start/policies/) for the current numbers).
- **Batch work** goes in a `Job`. **`sleep infinity` is bannable.**
- **Long-running services** use a `Deployment` (no GPU), auto-deleted after 2 weeks.
- **GPUs must run >40% utilized.** A100 quota is 0 by default and requires the [A100 access request](https://nrp.ai/documentation/userdocs/start/policies/).
- **Storage idle 6+ months is purged.**

## How NRP fits alongside other UW–Madison resources

For UW–Madison researchers and teams building LLM- or VLM-based applications, NRP complements rather than replaces other UW options. Two ways to think about the choice:

### Hosted LLM endpoints (call the model via API)

- **NRP** — free, open-weight models (Qwen3, GPT-OSS, Kimi, etc.) via OpenAI-compatible endpoint. Per-user concurrency limits; no service-level agreement; best-effort capacity.
- [**UW Cloud Services**](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/UW-Cloud-Services.html) — proprietary frontier via [Bedrock](https://aws.amazon.com/bedrock/) / [Gemini API](https://ai.google.dev/) / [Azure OpenAI](https://learn.microsoft.com/azure/ai-services/openai/); pay-per-token, fastest latency at a fee. UW-provisioned accounts cut grant overhead from 55.5% to 26%, get Internet2 NET+ pricing, and are HIPAA-eligible.
- [**GenAI at UW–Madison**](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/GenAI/GenAI-at-UW-Madison.html) — UW-vetted free chat UIs (Google Gemini, Microsoft 365 Copilot, NotebookLM). End-user tools, not programmatic APIs — useful for ad-hoc analysis, not for building applications.

### General compute (run your own code on GPUs)

- **NRP** — hosted [JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/), [Coder](https://nrp.ai/documentation/userdocs/coder/coder/), and self-managed GPU pods (RTX 2080 Ti / 3090 / 4090, A10, A100) on a free, multi-site Kubernetes cluster.
- [**CHTC**](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/CHTC.html) — free GPU Lab (A100s, H100s, H200s) on a traditional high-throughput computing model: borrow GPUs for limited-duration compute, with batching for jobs that need to run longer than a single allocation. Good for training, fine-tuning, parameter sweeps, and one-off inference up to ~70B with quantization on a single 80 GB GPU. Not a model-serving platform; no NVLink multi-GPU.
- [**UW Cloud Services**](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/UW-Cloud-Services.html) — hyperscaler hardware (B200s, NVLink multi-GPU, world-class latency) and managed ML platforms ([SageMaker](https://aws.amazon.com/sagemaker/), [Vertex AI](https://cloud.google.com/vertex-ai), [Azure ML](https://azure.microsoft.com/en-us/products/machine-learning)) for notebooks and training. Also the route for persistent endpoints that stay up beyond HPC job time limits. Same UW-account pricing benefits as above.

## Getting help

- **Matrix** (public rooms only): [nrp.ai/contact](https://nrp.ai/contact/)
- **Email**: [support@nationalresearchplatform.org](mailto:support@nationalresearchplatform.org)
- **Docs**: [nrp.ai/documentation](https://nrp.ai/documentation/) · [Getting Started](https://nrp.ai/documentation/userdocs/start/getting-started/) · [Fair-use policy](https://nrp.ai/documentation/userdocs/ai/llm-managed/fair-use/) · [Cluster Policies](https://nrp.ai/documentation/userdocs/start/policies/)
- **Reference**: Wang et al., *[The National Research Platform: Stretched, Multi-Tenant, Scientific Kubernetes Cluster](https://arxiv.org/abs/2505.22864)*, arXiv:2505.22864.

## Related resources

- [**Compute**: Center for High Throughput Computing (CHTC)](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/CHTC.html) — UW–Madison's free HPC/HTC platform; the default for batch training and parameter sweeps.
- [**Compute**: UW–Madison Cloud Services](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/UW-Cloud-Services.html) — AWS / GCP / Azure with UW-negotiated rates; the route for persistent endpoints, managed ML platforms, and NVLink multi-GPU.
- [**Compute**: BadgerCompute](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/BadgerCompute.html) — UW NetID-authenticated JupyterHub for short interactive sessions.
- [**Compute**: Google Colab](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/Compute/GoogleColab.html) — Free cloud-based Jupyter notebooks with GPU access for prototyping.
- [**Workshop/Compute**: Intro to GCP for Machine Learning & AI](https://uw-madison-datascience.github.io/ML-X-Nexus/Learn/Workshops/Intro-GCP.html) — Hands-on workshop covering Vertex AI, model training/tuning, and RAG pipelines on GCP.
- [**Workshop/Compute**: Intro to AWS SageMaker for Predictive ML/AI](https://uw-madison-datascience.github.io/ML-X-Nexus/Learn/Workshops/Intro-Amazon_SageMaker.html) — Workshop covering ML workflows in AWS SageMaker.
- [**GenAI**: UW Generative AI Services & Policies](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/GenAI/GenAI-at-UW-Madison.html) — UW-vetted enterprise chat and meeting AI tools (Google Gemini, M365 Copilot, NotebookLM, Zoom AI Companion, WebEx AI Assistant) and policy guidance.
- [**GenAI**: OpenScholar](https://uw-madison-datascience.github.io/ML-X-Nexus/Toolbox/GenAI/OpenScholar.html) — Open-source scientific literature synthesis tool; relevant for RAG-style workflows you could host on NRP.
- [**Notebook**: Quantization and Precision](https://uw-madison-datascience.github.io/ML-X-Nexus/Learn/Notebooks/Quantization-and-Precision.html) — Background on the quantization tricks that make large-model inference fit on a single GPU.
- [**Projects**: ML+X Machine Learning Marathon](https://uw-madison-datascience.github.io/ML-X-Nexus/Projects/ML-Marathon/) — Annual 12-week ML/AI hackathon at UW–Madison.

## Comments